In [3]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset --------
class RelativeSpeedDataset200D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            def smooth(x, w):
                return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items:
                    return

                d = dist[i:i+20]
                o = own[i:i+20]
                t = tgt[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)) or np.any(np.isnan(t)):
                    continue

                rel_speed = t - o
                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11)

                try:
                    feat = np.concatenate([
                        d, o, own_acc, d1, d2,
                        f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20]
                    ])
                except:
                    continue

                if feat.shape[0] != 200:
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return torch.tensor(feat), torch.tensor(tgt, dtype=torch.float32), sid

# -------- 改良版 LSTMモデル：全フレームの出力を平均 --------
class AvgFrameLSTMModel(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=256, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.3)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.view(x.size(0), 20, 10)       # (B, 20, 10)
        out, _ = self.lstm(x)               # (B, 20, hidden)
        out = self.fc1(out)                 # (B, 20, 64)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)                 # (B, 20, 1)
        return out.mean(dim=1).squeeze(1)   # 全時刻平均 (B,)

# -------- 学習ループ --------
def train_avgframe_lstm(dataset, save_path="model_avgframe_lstm.pth"):
    scenes = sorted(set([item[-1] for item in dataset.items]))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)

    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    def collate_fn(batch):
        feats, tgts, sids = zip(*batch)
        return torch.stack(feats), torch.tensor(tgts), list(sids)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AvgFrameLSTMModel().to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.SmoothL1Loss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, threshold=0.0001
    )

    best_val_loss = float('inf')
    patience = 40
    min_delta = 0.0005
    counter = 0

    for epoch in range(100):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if best_val_loss - val_loss > min_delta:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved model to {save_path} (val_loss={val_loss:.4f})")
            counter = 0
        else:
            counter += 1
            print(f"⏸ No significant improvement. Patience: {counter}/{patience}")
            if counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- 実行部 --------
if __name__ == "__main__":
    dataset = RelativeSpeedDataset200D(
        annot_root="../train/train_annotations",
        distance_json_path="../train2/distance1/corrected_distance_estimates_filtered.json",
        max_items=7500
    )

    model = train_avgframe_lstm(dataset, save_path="model_avgframe_lstm.pth")
    print("✅ 学習完了: model_avgframe_lstm.pth に保存しました")


[Train 1]: 100%|██████████| 92/92 [00:00<00:00, 154.22it/s]


Epoch 1 | Train Loss: 2.5797 | Val Loss: 0.2648
✅ Saved model to model_avgframe_lstm.pth (val_loss=0.2648)


[Train 2]: 100%|██████████| 92/92 [00:00<00:00, 197.10it/s]


Epoch 2 | Train Loss: 0.2897 | Val Loss: 0.1084
✅ Saved model to model_avgframe_lstm.pth (val_loss=0.1084)


[Train 3]: 100%|██████████| 92/92 [00:00<00:00, 191.25it/s]


...

Epoch 79 | Train Loss: 0.0475 | Val Loss: 0.0477
⏸ No significant improvement. Patience: 38/40


[Train 80]: 100%|██████████| 92/92 [00:00<00:00, 192.23it/s]


Epoch 80 | Train Loss: 0.0501 | Val Loss: 0.0508
⏸ No significant improvement. Patience: 39/40


[Train 81]: 100%|██████████| 92/92 [00:00<00:00, 192.56it/s]


Epoch 81 | Train Loss: 0.0533 | Val Loss: 0.0486
⏸ No significant improvement. Patience: 40/40
🛑 Early stopping at epoch 81
✅ 学習完了: model_avgframe_lstm.pth に保存しました


In [1]:
import os
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

# -------- モデル定義（学習時と同じ構造） --------
class AvgFrameLSTMModel(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=256, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.3)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.view(x.size(0), 20, 10)
        out, _ = self.lstm(x)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out.mean(dim=1).squeeze(1)

# -------- 推論用 Dataset --------
class InferenceDataset200D(Dataset):
    def __init__(self, annot_root, distance_json_path):
        self.items = []
        self.seq_lens = {}

        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            self.seq_lens[sid] = len(seq)

            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            def smooth(x, w):
                return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                d = dist[i:i+20]
                o = own[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)):
                    continue

                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)

                feat = np.concatenate([
                    d, o, own_acc, d1, d2,
                    f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20]
                ])

                if feat.shape[0] != 200:
                    continue

                own_avg = np.mean(o)
                self.items.append((feat.astype(np.float32), own_avg, sid, i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, own_avg, sid, frame_idx = self.items[idx]
        return torch.tensor(feat), own_avg, sid, frame_idx

# -------- 推論処理 + submission.json 作成 --------
def predict_and_save_submission(
    model_path,
    annot_root,
    distance_json_path,
    save_path="submission.json"
):
    dataset = InferenceDataset200D(annot_root, distance_json_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AvgFrameLSTMModel().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    raw_preds = defaultdict(list)
    with torch.no_grad():
        for feats, own_speeds, sids, frame_idxs in tqdm(loader):
            feats = feats.to(device)
            preds = model(feats).cpu().numpy()
            own_speeds = own_speeds.numpy()
            abs_speeds = preds + own_speeds

            for sid, frame_idx, tgt in zip(sids, frame_idxs, abs_speeds):
                raw_preds[sid].append((frame_idx + 19, float(round(tgt, 3))))

    submission = {}
    for sid, pairs in raw_preds.items():
        pairs.sort()
        seq_len = dataset.seq_lens.get(sid, max(f for f, _ in pairs) + 1)
        pred_list = [0.0] * seq_len
        for idx, val in pairs:
            if idx < seq_len:
                pred_list[idx] = val
        for i in range(1, seq_len):
            if pred_list[i] == 0.0:
                pred_list[i] = pred_list[i - 1]
        submission[sid] = pred_list

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    print(f"✅ 完成: {save_path} に保存しました（scene数: {len(submission)}）")

# -------- 実行部 --------
if __name__ == "__main__":
    predict_and_save_submission(
        model_path="model_avgframe_lstm.pth",
        annot_root="../test/test_annotations",
        distance_json_path="../testdistance/testdistance_estimates_smoothed.json",
        save_path="submission.json"
    )


100%|██████████| 395/395 [00:01<00:00, 261.91it/s]


✅ 完成: submission.json に保存しました（scene数: 239）


In [7]:
import json
import numpy as np

# ===== 設定 =====
INPUT_PATH = "submission.json"  # または "submission_smoothed.json"
OUTPUT_PATH = "submission_smoothed.json"
WINDOW_SIZE = 3
START_INDEX = 19  # 0-based index（20フレーム目から）

# ===== 読み込み =====
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    submission = json.load(f)

# ===== 平滑化処理 =====
smoothed_submission = {}

for scene_id, values in submission.items():
    values_np = np.array(values, dtype=float)
    smoothed = list(values_np[:START_INDEX])  # 先頭はそのまま保持

    for i in range(START_INDEX, len(values_np)):
        left = max(START_INDEX, i - WINDOW_SIZE // 2)
        right = min(len(values_np), i + WINDOW_SIZE // 2 + 1)
        avg = np.mean(values_np[left:right])
        smoothed.append(round(avg, 4))

    smoothed_submission[scene_id] = smoothed

# ===== 保存 =====
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(smoothed_submission, f, indent=2)

print(f"✅ 20フレーム目以降を平滑化した結果を '{OUTPUT_PATH}' に保存しました。")


✅ 20フレーム目以降を平滑化した結果を 'submission_smoothed.json' に保存しました。
